In [ ]:
# Disable TensorFlow to avoid potential DLL issues and use PyTorch instead
import os
os.environ['USE_TF'] = 'NO'
os.environ['USE_TORCH'] = 'YES'

# Importing the libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)
import torch
from torch.utils.data import Dataset
import gc

In [2]:
# Load the local IMDB dataset
df = pd.read_csv('IMDB Dataset.csv')
df.head(5)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
# Create train and test splits (80-20 split)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [4]:
class IMDBDatasetForTrainer(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.reviews = df['review'].values
        self.sentiments = df['sentiment'].values
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        review = str(self.reviews[idx])
        label = 1 if self.sentiments[idx] == 'positive' else 0

        # Tokenize the review
        encoding = self.tokenizer(
            review,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'][0],
            'attention_mask': encoding['attention_mask'][0],
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [5]:
# Define metrics computation function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}

#### **Benefits of Freezing base model parameters:**

1. **Faster Training**: Only the classifier layers train, so training is much faster
2. **Less Memory Usage**: Fewer parameters to update means lower GPU memory requirements
3. **Prevents Overfitting**: The pre-trained features are preserved, reducing risk of overfitting on small datasets
4. **Stable Features**: The base model's learned representations stay intact
5. **Good for Small Datasets**: When you have limited training data, freezing prevents destroying the pre-trained knowledge


In [6]:
# Initialize tokenizer and create datasets
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
train_dataset = IMDBDatasetForTrainer(train_df, tokenizer)
test_dataset = IMDBDatasetForTrainer(test_df, tokenizer)

# Initialize model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", 
    num_labels=2
)

# Freeze base model parameters
for param in model.distilbert.parameters():
    param.requires_grad = False

model.classifier

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [10]:
# Define base directory and model name
BASE_DIR = "./data"
MODEL_NAME = "sentiment_analysis"

# Create output directory
output_dir = os.path.join(BASE_DIR, MODEL_NAME)
os.makedirs(output_dir, exist_ok=True)

# Initialize training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    gradient_accumulation_steps=8,
    optim="adamw_torch",
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    report_to="none",
    logging_steps=500,
    disable_tqdm=False
)

In [11]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# Enable garbage collection
gc.enable()

# Train the model
print("Starting training...")
trainer.train()

# Clean up memory
gc.collect()
torch.cuda.empty_cache()

Starting training...


C:\Users\Paschal Alaemezie\AppData\Roaming\Python\Python312\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.445400,0.413965,0.808200


In [12]:
# Evaluate the model
print("\nEvaluating model...")
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")


Evaluating model...


Evaluation Results: {'eval_loss': 0.4139651358127594, 'eval_accuracy': 0.8082, 'eval_runtime': 170.2305, 'eval_samples_per_second': 58.744, 'eval_steps_per_second': 14.686, 'epoch': 1.0}


In [17]:
formatted_results = {
    'Metric': ['Loss', 'Accuracy', 'Runtime (seconds)', 'Samples/second', 'Steps/second', 'Epoch'],
    'Value': [
        f"{eval_results['eval_loss']:.4f}",
        f"{eval_results['eval_accuracy']:.4f} ({eval_results['eval_accuracy']:.1%})",
        f"{eval_results['eval_runtime']:.2f}",
        f"{eval_results['eval_samples_per_second']:.1f}",
        f"{eval_results['eval_steps_per_second']:.3f}",
        f"{eval_results['epoch']:.0f}"
    ]
}

# Create and display DataFrame
results_df = pd.DataFrame(formatted_results)
print("\n📊 Evaluation Results:")
print("=" * 40)
print(results_df.to_string(index=False))
print("=" * 40)


📊 Evaluation Results:
           Metric          Value
             Loss         0.4140
         Accuracy 0.8082 (80.8%)
Runtime (seconds)         170.23
   Samples/second           58.7
     Steps/second         14.686
            Epoch              1


In [13]:
# Save the model and tokenizer
model_save_path = "./data/sentiment_analysis"
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print("Model and tokenizer saved!")

Model and tokenizer saved!


In [14]:
def predict_sentiment(model, tokenizer, text, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """
    Predict sentiment for a given text with confidence score
    
    Args:
        model: The trained model
        tokenizer: The tokenizer used for the model
        text: Text input for sentiment analysis
        device: Device to run the model on ('cuda' or 'cpu')
    
    Returns:
        dict: Dictionary containing sentiment prediction, probability, and input text
    """
    model.eval()
    # Tokenize the text
    encoded_text = tokenizer(
        text,
        max_length=512,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    
    # Move to device
    input_ids = encoded_text['input_ids'].to(device)
    attention_mask = encoded_text['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=1)
        prediction = torch.argmax(probabilities, dim=1)
        confidence_score = probabilities[0][prediction].item()
    
    result = {
        'text': text,
        'sentiment': 'positive' if prediction.item() == 1 else 'negative',
        'confidence': f"{confidence_score:.2%}"
    }
    
    return result

In [15]:
# Example usage:
# Single prediction
text = "This movie was really great! I enjoyed every minute of it."
result = predict_sentiment(model, tokenizer, text)
prediction_df = pd.DataFrame([result])
display(prediction_df)


# Multiple predictions
texts = [
    "This movie was really great! I enjoyed every minute of it.",
    "I wouldn't recommend this movie to anyone. It was terrible."
]

results = []
for text in texts:
    result = predict_sentiment(model, tokenizer, text)
    results.append(result)

# Create DataFrame with results
predictions_df = pd.DataFrame(results)
print("\nMultiple Predictions:")
display(predictions_df)

,text,sentiment,confidence
0,This movie was really great! I enjoyed every m...,positive,99.70%



Multiple Predictions:


,text,sentiment,confidence
0,This movie was really great! I enjoyed every m...,positive,99.70%
1,I wouldn't recommend this movie to anyone. It ...,negative,95.64%
